In [64]:
import pandas as pd
import json

# 1. Ne pas couper les colonnes (les forcer à rester sur une seule ligne)
pd.set_option('display.expand_frame_repr', False)

# 2. Afficher toutes les colonnes (remplace les "...")
pd.set_option('display.max_columns', None)

# 3. Limiter la largeur du texte dans les cellules 
# (évite qu'une cellule de 500 caractères ne décale tout le tableau)
pd.set_option('display.max_colwidth', 50) 

# 4. Ajuster la précision des chiffres (optionnel)
pd.set_option('display.precision', 2)

In [65]:
"""# 1. Charger ton fichier (remplace 'data.json' par ton nom de fichier)
with open('conversations_turns.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
# 2. "Aplatir" le JSON
# record_path : on descend dans la liste 'Turns'
# meta : on garde les infos de haut niveau pour chaque ligne
df = pd.json_normalize(
    data, 
    record_path=['Turns'], 
    meta=['Id', 'Title', 'Url', 'Platform']
)
# 3. Nettoyer les noms de colonnes
# Par défaut, Pandas va créer 'Turn.User', 'Turn.Assistant', etc.
# On peut les renommer pour plus de clarté :
df.columns = [c.replace('Turn.', '') for c in df.columns]
# Afficher un aperçu
print(df)"""

'# 1. Charger ton fichier (remplace \'data.json\' par ton nom de fichier)\nwith open(\'conversations_turns.json\', \'r\', encoding=\'utf-8\') as f:\n    data = json.load(f)\n# 2. "Aplatir" le JSON\n# record_path : on descend dans la liste \'Turns\'\n# meta : on garde les infos de haut niveau pour chaque ligne\ndf = pd.json_normalize(\n    data, \n    record_path=[\'Turns\'], \n    meta=[\'Id\', \'Title\', \'Url\', \'Platform\']\n)\n# 3. Nettoyer les noms de colonnes\n# Par défaut, Pandas va créer \'Turn.User\', \'Turn.Assistant\', etc.\n# On peut les renommer pour plus de clarté :\ndf.columns = [c.replace(\'Turn.\', \'\') for c in df.columns]\n# Afficher un aperçu\nprint(df)'

In [66]:
import pandas as pd
import json

# 1. Charger le JSON
with open('conversations.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Aplatir le JSON
df = pd.json_normalize(
    data, 
    record_path=['Turns'], 
    meta=['Id', 'Title', 'Url', 'Platform']
)
df.columns = [c.replace('Turn.', '') for c in df.columns]

#print(df)

In [67]:
# Dataset 1 : métadonnées des conversations (une ligne par conversation)
df_conversations = df[['Id', 'Title', 'Url', 'Platform']].drop_duplicates()

# Dataset 2 : les turns (une ligne par turn, Id comme clé étrangère)
df_turns = df[['Id', 'Turn_id', 'User', 'Assistant']]

# Nettoyage des colonnes User et Assistant
import re
clean_regex = r"^(?:Vous avez dit|ChatGPT a dit|Copilot a dit)\s*:?\s*"
reflexion_regex = r"^Réflexion durant \d+(?:m \d+)?s\s*"

# Application sur df_turns
df_turns = df_turns.copy()
df_turns['User'] = df_turns['User'].str.replace(clean_regex, "", regex=True, flags=re.IGNORECASE).str.strip()
df_turns['User'] = df_turns['User'].str.replace(r'Fichier chargé\s*', '', regex=True).str.strip()
df_turns['Assistant'] = df_turns['Assistant'].str.replace(clean_regex, "", regex=True, flags=re.IGNORECASE).str.strip()
df_turns['Assistant'] = df_turns['Assistant'].str.replace(reflexion_regex, "", regex=True).str.strip()

print(df_conversations ,'\n')
print(df_turns)


      Id                              Title                                                Url Platform
0      1                             Claude  https://claude.ai/share/d9c63141-e1d0-4044-978...   claude
2      2                             Claude  https://claude.ai/share/8f850ae8-920c-461c-af9...   claude
5      3                             Claude  https://claude.ai/share/c826dc67-e832-44f4-b9e...   claude
6      4                             Claude  https://claude.ai/share/932df61d-6cad-4b92-bb1...   claude
7      5                             Claude  https://claude.ai/share/e6737bec-520e-4220-acc...   claude
...   ..                                ...                                                ...      ...
1521  84  Préparation quiz machine learning  https://chatgpt.com/share/68efa949-1b94-8011-9...  chatgpt
1628  85          Hackathon support request  https://chatgpt.com/share/68efac53-19d8-8003-a...  chatgpt
1739  86          File analysis preparation  https://chatgpt.com

In [68]:
# La fonction Flush est très importante, c'est elle qui détérmine si on regarde tout le bloc de code ou pas pour tester si python ou bash
# On crée un bloc de code python dès que le regex est satisfait, on ferme le bloc dès qu'il y'a un saut de ligne
import re

# Fusion des blocs python consécutifs séparés par une ligne vide
def merge_consecutive_blocks(result_parts):
    changed = True
    while changed:
        changed = False
        new_parts = []
        i = 0
        while i < len(result_parts):
            tag1 = result_parts[i].split('\n')[0] if result_parts[i].startswith('```') else None
            tag2 = result_parts[i+2].split('\n')[0] if i + 2 < len(result_parts) and result_parts[i+2].startswith('```') else None
            if (i + 2 < len(result_parts)
                    and tag1 and tag2
                    and tag1 == tag2
                    and result_parts[i+1] == ''):
                content1 = result_parts[i][len(tag1)+1:-len('\n```')]
                content2 = result_parts[i+2][len(tag2)+1:-len('\n```')]
                new_parts.append(f"{tag1}\n{content1}\n\n{content2}\n```")
                i += 3
                changed = True
            else:
                new_parts.append(result_parts[i])
                i += 1
        result_parts = new_parts
    return result_parts

# Sert à gérer les cas où on a un :
def split_inline_code(line):
    inline_triggers = r'(\bimport\s|\bfrom\s[\w.]+\s+import|\bdef\s|\bclass\s|\bprint\s*\()'
    match = re.search(inline_triggers, line)
    if match and match.start() > 0:
        return [line[:match.start()].rstrip(), line[match.start():]]
    return [line]

# Crée les blocs de code
def format_code_blocks(texte):
    pattern = re.compile(r"(?:"
        # Python - uniquement en début de ligne
        r"^\s*\bimport\s|^\s*\bfrom\s[\w.]|^\s*\bdef\s|^\s*\bclass\s"
        r"|^\s*\bfor\s+\w+\s+in\s|^\s*\bwhile\s|^\s*\bif\s|^\s*\belif\s|^\s*\belse:"
        r"|^\s*\breturn\s|^\s*\byield\s|^\s*\braise\s"
        r"|^\s*\bwith\s|^\s*\btry:|^\s*\bexcept\s|^\s*\bfinally:"
        r"|^\s*[a-zA-Z_]\w*[\w\s,]*=\s*[^\s=!<>]"
        r"|^\s*[a-zA-Z_]\w*\s*[\+\-\*\/]="
        r"|^\s*\bprint\s*\("
        r"|^\s*#.*"
        # Bash / erreurs
        r"|^\s*\$\s*\w+"                          # commandes shell : $ git, $ pip...
        r"|^\s*\bpip\s|^\s*\bconda\s|^\s*\bgit\s"
        r"|^\s*\bsudo\s|^\s*\bapt\s"
        r"|^\s*\becho\s|^\s*\bexport\s"
        r"|Traceback|KeyboardInterrupt"
        r"|Error\b|Warning\b|Exception\b"
        r"|File\s+\"|line\s+\d+"
        r"|-{3,}"
        r")", re.MULTILINE)


    lines = texte.split('\n')
    # Sert à gérer les cas où on a un :
    lines = [sub for line in lines for sub in split_inline_code(line)]
    result_parts = []
    current_code_lines = []
    in_code_block = False

    def detect_lang(code):
        if re.search(r"^\s*-{3,}\s*$", code):  # ligne contenant uniquement des tirets
            return "bash"
        bash_kws = ["Traceback", "KeyboardInterrupt", "Error", "Warning", "Exception",
                    "File \"", "echo ", "export ", "sudo ", "apt ", "pip ", "conda ",
                    "git ", "docker ", "chmod ", "grep ", "curl ", "wget ",
                    " | ", ">>", "$", "c:\\"]
        for kw in bash_kws:
            if kw in code:
                return "bash"
        return "python"

    def flush_code():
        if current_code_lines:
            first_line = current_code_lines[0]
            lang = detect_lang(first_line)
            code = '\n'.join(current_code_lines)
            result_parts.append(f"```{lang}\n{code}\n```")
            current_code_lines.clear()

    for line in lines:
        if in_code_block:
            if line.strip() == '':
                # Saut de ligne → on ferme le bloc
                flush_code()
                in_code_block = False
                result_parts.append('')
            else:
                current_code_lines.append(line)
        else:
            if pattern.search(line):
                # Pattern trouvé → on ouvre un nouveau bloc
                in_code_block = True
                current_code_lines.append(line)
            else:
                result_parts.append(line)

    # Fermer le dernier bloc si on est encore dedans
    if in_code_block:
        flush_code()
        result_parts.append("")  # ✅ force une vraie séparation

    return '\n'.join(merge_consecutive_blocks(result_parts))

# --- TEST ---
# On cherche les lignes où la colonne 'Id' vaut 12
#input_sale = df_turns[df_turns['Id'] == 12]["User"].iloc[23]
#print(input_sale)
df_turns["User_tagged"] = df_turns["User"].apply(format_code_blocks)
print(df_turns["User_tagged"])


0       Est ce que c'est possible de partager un proje...
1       fait moi un liste de tout les algorithme de ma...
2       Peux tu me faire le code : Your task is to imp...
3       Est ce que ce code est juste :\n```python\ndef...
4       Tick the correct boxes for the decision trees:...
                              ...                        
1820    ```python\n# Random Forests\r\nfrom sklearn.en...
1821    ```python\n# === Titanic — v3 (FE++ + KFold Ta...
1822        comment add tout les fichiers d'un repertoire
1823                                         comment push
1824    PS C:\Users\Admin\Desktop\hackathon-ml-2025-gr...
Name: User_tagged, Length: 1825, dtype: str


In [69]:
# On essaie de compléter les DataSets
# 1 - On compte le nombre de tokens dans la colonne 'User'
import spacy
nlp = spacy.load("fr_core_news_sm")

#df_turns["n_tokens"] = df_turns["User"].apply(lambda x: len(nlp(str(x))))
df_turns["n_words"] = df_turns["User"].apply(lambda x: len(str(x).split()))

#Stop words
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

stop_words = set(stopwords.words('french'))  # ou 'french'
df_turns["n_stopwords"] = df_turns["User"].apply(
    lambda x: sum(1 for word in str(x).lower().split() if word in stop_words)
)

""""
# Sentiment analysis
from transformers import pipeline

sentiment = pipeline(
    "sentiment-analysis",
    model="tabularisai/multilingual-sentiment-analysis"
)

df_turns["sentiment"] = df_turns["User"].apply(
    lambda x: sentiment(str(x)[:512])[0]["label"]
) 
"""

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rayan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


'"\n# Sentiment analysis\nfrom transformers import pipeline\n\nsentiment = pipeline(\n    "sentiment-analysis",\n    model="tabularisai/multilingual-sentiment-analysis"\n)\n\ndf_turns["sentiment"] = df_turns["User"].apply(\n    lambda x: sentiment(str(x)[:512])[0]["label"]\n) \n'

In [70]:
# Sentiments    
from textblob import TextBlob
from textblob_fr import PatternTagger, PatternAnalyzer

df_turns["sentiment"] = df_turns["User"].apply(
    lambda x: TextBlob(str(x), pos_tagger=PatternTagger(), analyzer=PatternAnalyzer()).sentiment[0]
)

print(df_turns)


      Id  Turn_id                                               User                                          Assistant                                        User_tagged  n_words  n_stopwords  sentiment
0      1        1  Est ce que c'est possible de partager un proje...  Oui, il est possible de partager un projet Cla...  Est ce que c'est possible de partager un proje...       11            5       0.00
1      1        2  fait moi un liste de tout les algorithme de ma...  Je vais vous créer une analyse comparative des...  fait moi un liste de tout les algorithme de ma...       44           21       0.26
2      2        1  Peux tu me faire le code : Your task is to imp...  Voici l'implémentation complète de la régressi...  Peux tu me faire le code : Your task is to imp...      198            6      -0.47
3      2        2  Est ce que ce code est juste :def gradient_des...  Oui, ce code est parfaitement juste ! 👍 Il imp...  Est ce que ce code est juste :\n```python\ndef...      147     

In [71]:
# On crée deux nouvelles colonnes pour indiquer la présence de code Python ou Bash

df_turns["Python"] = 0
df_turns["Bash"] = 0
for idx, sentence in df_turns["User_tagged"].items():
    if "```python" in sentence:
        df_turns.at[idx, "Python"] = 1
    if "```bash" in sentence:  # if et non elif, une phrase peut avoir les deux
        df_turns.at[idx, "Bash"] = 1

#print(df_turns[["User_tagged", "Python", "Bash"]])

print(df_turns["User_tagged"].iloc[1])


fait moi un liste de tout les algorithme de machine learning du plus performant au moins. Fait de même du plus rapide au moins rapide et fait un classement de celui qui est le plus rapide et le plus performant à la fois !


In [72]:
# Longueur moyenne des mots 
for id in df_conversations["Id"]:
    df_conversations.loc[df_conversations["Id"] == id, "average_words"] = df_turns[df_turns["Id"] == id]["n_words"].mean()

# Nombre totale de mots dans la conversation
total_words = df_turns.groupby("Id")["n_words"].sum().reset_index()
total_words.columns = ["Id", "total_words"]

df_conversations = df_conversations.merge(total_words, on="Id", how="left")
print(df_conversations)
print(df_conversations.head())

    Id                              Title                                                Url Platform  average_words  total_words
0    1                             Claude  https://claude.ai/share/d9c63141-e1d0-4044-978...   claude          27.50           55
1    2                             Claude  https://claude.ai/share/8f850ae8-920c-461c-af9...   claude         144.67          434
2    3                             Claude  https://claude.ai/share/c826dc67-e832-44f4-b9e...   claude          36.00           36
3    4                             Claude  https://claude.ai/share/932df61d-6cad-4b92-bb1...   claude          51.00           51
4    5                             Claude  https://claude.ai/share/e6737bec-520e-4220-acc...   claude          14.00           14
..  ..                                ...                                                ...      ...            ...          ...
68  84  Préparation quiz machine learning  https://chatgpt.com/share/68efa949-1b94-8011-9.

In [73]:
"""
# On import les notes dU prof
import pandas as pd

read = pd.read_excel("../../Project_Submissions/NS - Grades.xlsx")
read.to_csv("Notes.csv", index=None, header = False)
notes = pd.DataFrame(pd.read_csv("Notes.csv"))
#print(notes)

# Feedback into notes je crois
read2 = pd.read_excel("../../Project_Submissions/Feedback on Collaboration and sending the Chat Link.xlsx")
read2.to_csv("Groupes.csv", index=None, header = True)
tmp = pd.DataFrame(pd.read_csv("Groupes.csv"))

# Ass
# Faux 
groupes["nb","url"] = tmp["What is the group number ?","Paste here the link of the chat you used during this Hackathon:"]

"""

'\n# On import les notes dU prof\nimport pandas as pd\n\nread = pd.read_excel("../../Project_Submissions/NS - Grades.xlsx")\nread.to_csv("Notes.csv", index=None, header = False)\nnotes = pd.DataFrame(pd.read_csv("Notes.csv"))\n#print(notes)\n\n# Feedback into notes je crois\nread2 = pd.read_excel("../../Project_Submissions/Feedback on Collaboration and sending the Chat Link.xlsx")\nread2.to_csv("Groupes.csv", index=None, header = True)\ntmp = pd.DataFrame(pd.read_csv("Groupes.csv"))\n\n# Ass\n# Faux \ngroupes["nb","url"] = tmp["What is the group number ?","Paste here the link of the chat you used during this Hackathon:"]\n\n'

In [74]:
# On ajoute les images dans la colonne image et on les supprimes de la colonne User

import re

df_turns["img"] = df_turns["User"].apply(
    lambda text: re.findall(r'images\\[^\]]+', str(text))
)

df_turns["User"] = df_turns["User"].apply(
    lambda text: re.sub(r'\[images[^\]]+\]', '', str(text)).strip()
)

df_turns["User_tagged"] = df_turns["User_tagged"].apply(
    lambda text: re.sub(r'\[images[^\]]+\]', '', str(text)).strip()
)

# On affiche les lignes où il y a des images
print(df_turns[df_turns["img"].apply(len) > 0])

# Remarque : il y'a parfois des textes vides. Ceci est du au fait que l'utilisateur a juste chargé des images, sans écrire du texte.

      Id  Turn_id                                               User                                          Assistant                                        User_tagged  n_words  n_stopwords  sentiment  Python  Bash                                                img
26     9        2              ok je veux faire ca :\nl'option B stp  Parfait 🌺 ! Tu veux donc suivre l’Option B : L...              ok je veux faire ca :\nl'option B stp       10            1       0.00       0     0                        [images\img_9_2_user_1.jpg]
32     9        8    ok super on va commencer a faire les cellules :  Parfait 🌺 ! On attaque la partie la plus sympa...    ok super on va commencer a faire les cellules :       11            2       0.70       0     0                        [images\img_9_8_user_1.jpg]
33     9        9                                                     Exactement 🌺 — tu vois, l’erreur est parfaitem...                                                           1            0 

In [75]:
# On ajoute la colonne Group aux conversations
feedback = pd.read_excel("../../Project_Submissions/Feedback on Collaboration and sending the Chat Link.xlsx")
url_to_group = {}
for _, row in feedback.iterrows():
    urls = re.findall(r'https?://\S+', str(row["Paste here the link of the chat you used during this Hackathon:"]))
    urls = [url for url in urls if "share" in url]
    for url in urls:
        url_to_group[url] = str(row["What is the group number ?"]).replace("Group ", "").strip()
df_conversations["group"] = df_conversations["Url"].map(url_to_group)

# On ajoute la colonne grade aux conversations
notes = pd.read_excel("../../Project_Submissions/NS - Grades.xlsx", header=1)
group_to_grade = dict(zip(notes["Project Group"].astype(str).str.strip(), notes["Projet Titanic Total"]))
group_to_quiz = dict(zip(notes["Project Group"].astype(str).str.strip(), notes["Quiz: Quizz (Phase 1)"]))
group_to_code = dict(zip(notes["Project Group"].astype(str).str.strip(), notes["Code Analysis"]))

df_conversations["quiz"] = df_conversations["group"].map(group_to_quiz)
df_conversations["code_analysis"] = df_conversations["group"].map(group_to_code)
df_conversations["grade"] = df_conversations["group"].map(group_to_grade)

# On ajoute la colonne nb_turns aux conversations
nb_turns = df_turns.groupby("Id")["Turn_id"].count().reset_index()
nb_turns.columns = ["Id", "nb_turns"]

df_conversations = df_conversations.merge(nb_turns, on="Id", how="left")
print(max(notes["Code Analysis"]))

nan


In [76]:
# Export the datasets
df_turns.to_csv("df_turns.csv", index=False)
df_conversations.to_csv("df_conversations.csv", index=False)

In [77]:
print(df_turns)

      Id  Turn_id                                               User                                          Assistant                                        User_tagged  n_words  n_stopwords  sentiment  Python  Bash img
0      1        1  Est ce que c'est possible de partager un proje...  Oui, il est possible de partager un projet Cla...  Est ce que c'est possible de partager un proje...       11            5       0.00       0     0  []
1      1        2  fait moi un liste de tout les algorithme de ma...  Je vais vous créer une analyse comparative des...  fait moi un liste de tout les algorithme de ma...       44           21       0.26       0     0  []
2      2        1  Peux tu me faire le code : Your task is to imp...  Voici l'implémentation complète de la régressi...  Peux tu me faire le code : Your task is to imp...      198            6      -0.47       1     0  []
3      2        2  Est ce que ce code est juste :def gradient_des...  Oui, ce code est parfaitement juste ! 👍 Il